# Global Run

Runs batch green-ammonia optimisation over all locations in the max-capacities
CSV and applies per-location spatial cost inputs (offshore multipliers,
interest rates, etc.).

> **Local vs HPC guidance**
>
> Each location takes ~6 s at full hourly resolution (8 760 snapshots).
> The full global grid (~54 000 cells) would take ~90 hours serially.
>
> * **Locally**: run a small subset (set `LOCATIONS` or `LOCATIONS_CSV`,
>   or lower `MAX_SNAPSHOTS` for smoke tests).
> * **On ARC**: use `arc/submit_global_run.sh <label> --quadrants` to submit
>   4 parallel longitude-segment jobs. See [arc/README.md](../arc/README.md).
> * After ARC jobs complete, use the **"Combine segmented results"** cell at
>   the bottom to merge quadrant CSVs before analysing in notebook 05.

In [ ]:
from __future__ import annotations

import importlib
import os
import sys
from pathlib import Path

import pandas as pd
import yaml


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'model').is_dir():
            return p
    raise RuntimeError("Could not locate repo root containing 'model/'")


NOTEBOOK_DIR = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
repo_root = find_repo_root(NOTEBOOK_DIR)

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from model import auxiliary as aux
from model import main as plant_main
from model import run_global
from model import plot_global_heatmap

importlib.reload(aux)
importlib.reload(plant_main)
importlib.reload(run_global)
importlib.reload(plot_global_heatmap)

In [ ]:
# ----------------
# USER PARAMETERS
# ----------------
LOCATIONS = None           # None = all locations from MAX_CAPACITIES_CSV
LOCATIONS_CSV = None       # or provide a CSV with lat,lon columns

TECH_YAML = repo_root / 'inputs' / 'tech_config_ammonia_plant_2030_dea.yaml'
MAX_CAPACITIES_CSV = repo_root / 'data' / '20251222_max_capacities.csv'
SPATIAL_COST_CSV = repo_root / 'inputs' / 'spatial_cost_inputs.csv'

try:
    tech_meta = yaml.safe_load(Path(TECH_YAML).read_text()) or {}
    TECH_CURRENCY = str(tech_meta.get('currency', '')).strip().upper() or None
except Exception as exc:
    TECH_CURRENCY = None
    print('Warning: could not read currency from tech YAML:', exc)

if TECH_CURRENCY:
    os.environ['GREEN_LORY_CURRENCY'] = TECH_CURRENCY
    print('Tech-config currency:', TECH_CURRENCY)
else:
    print('Tech-config currency: (not set)')

os.environ['GREEN_LORY_SOLVER'] = 'gurobi'
os.environ.setdefault('GREEN_LORY_SOLVER_LOG', '0')
print('Solver:', os.environ['GREEN_LORY_SOLVER'])

AGGREGATION_COUNT = 1
TIME_STEP_H = 1.0
MAX_SNAPSHOTS = None       # None = full 8760; set to e.g. 168 for quick tests
QUIET = True

# Solver thread count
THREADS_PER_WORKER = os.cpu_count() or 1
print(f'Solver threads: {THREADS_PER_WORKER}')

OUTPUT_DIR = repo_root / 'results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = OUTPUT_DIR / 'global_run_results.csv'
HEATMAP_HTML = OUTPUT_DIR / 'global_lcoa_heatmap.html'

COLOR_COLUMN = (f'lcoa_{TECH_CURRENCY.lower()}_per_t' if TECH_CURRENCY else 'lcoa_usd_per_t')
COLOR_SCALE = 'Cividis'

In [ ]:
if LOCATIONS_CSV is not None:
    locations = run_global.load_locations_csv(LOCATIONS_CSV)
else:
    locations = LOCATIONS

print('Locations mode:', 'ALL (from max-capacities CSV)' if locations is None else f'{len(locations)} explicit locations')
print('Spatial cost inputs:', SPATIAL_COST_CSV if SPATIAL_COST_CSV.exists() else '(not found — will use YAML defaults)')

In [ ]:
import time as _time

_t0 = _time.perf_counter()

results_df = run_global.run_global(
    locations=locations,
    aggregation_count=AGGREGATION_COUNT,
    time_step=TIME_STEP_H,
    max_snapshots=MAX_SNAPSHOTS,
    output_csv=RESULTS_CSV,
    tech_yaml=TECH_YAML,
    land_csv=MAX_CAPACITIES_CSV,
    interest_csv=SPATIAL_COST_CSV,
    quiet=QUIET,
    threads_per_worker=THREADS_PER_WORKER,
)

_elapsed = _time.perf_counter() - _t0
print(f'Finished: {len(results_df)} locations in {_elapsed:.0f}s ({_elapsed / 60:.1f} min)')
print('Saved CSV to:', RESULTS_CSV)
display(results_df.head())

In [ ]:
fig = plot_global_heatmap.plot_lcoa_heatmap(
    RESULTS_CSV,
    color_column=COLOR_COLUMN,
    cell_size_deg=1.0,
    color_scale=COLOR_SCALE,
)

fig.show()
fig.write_html(HEATMAP_HTML)
print('Saved heatmap HTML to:', HEATMAP_HTML)

## Combine segmented results

Use this section after ARC quadrant jobs complete. It reads all quadrant CSVs
from `results/<run-label>-*/`, merges them, deduplicates, and writes a single
combined output ready for analysis in notebook 05.

In [ ]:
# Combine quadrant CSV files into a single results table.
# USER: set COMBINE_RUN_LABEL to the run-label used for submission.
import glob

COMBINE_RUN_LABEL = 'global-dea-flat-2030'  # <-- edit to match your ARC run label

quadrant_pattern = str(OUTPUT_DIR / f'{COMBINE_RUN_LABEL}-*' / 'run_global_*.csv')
quadrant_files = sorted(glob.glob(quadrant_pattern))

if not quadrant_files:
    print(f'No quadrant CSVs found matching: {quadrant_pattern}')
    print('Check that your ARC jobs have completed and results are downloaded.')
else:
    print(f'Found {len(quadrant_files)} quadrant file(s):')
    for f in quadrant_files:
        print(f'  {f}')

    parts = [pd.read_csv(f) for f in quadrant_files]
    combined = pd.concat(parts, ignore_index=True)

    # Deduplicate by (latitude, longitude), keeping the last occurrence.
    before = len(combined)
    combined = combined.drop_duplicates(subset=['latitude', 'longitude'], keep='last')
    combined = combined.sort_values(['latitude', 'longitude']).reset_index(drop=True)
    after = len(combined)
    if before != after:
        print(f'Deduplicated: {before} -> {after} rows')

    COMBINED_CSV = OUTPUT_DIR / f'{COMBINE_RUN_LABEL}_combined.csv'
    combined.to_csv(COMBINED_CSV, index=False)
    print(f'Combined {after} locations -> {COMBINED_CSV}')
    display(combined.head())